# Decision Logic Layer

Turns the price, weather and demand data into four farmer-facing signals: sell-now-vs-wait, arbitrage / best market, demand level, and anomaly flag.



In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

data = pd.read_csv('market_data_with_weather.csv')
data['date'] = pd.to_datetime(data['date'])

print(data.shape)

(21107, 14)


## 1. Feature Engineering
Lags at 1, 2, 3 and 12 months -- the 12-month lag captures annual seasonality that shorter lags miss. `demand_index` is included as a feature: testing confirmed it improves XGBoost by 2.12 MAPE points.

In [ ]:
TODAY = pd.Timestamp.today().normalize()

price_ml_base = data[data['forecastable'] == True].copy()
price_ml_base = price_ml_base.sort_values(by=['market', 'commodity', 'date'])

for lag in [1, 2, 3, 12]:
    price_ml_base[f'price_lag_{lag}'] = price_ml_base.groupby(['market', 'commodity'])['price'].shift(lag)
    price_ml_base[f'rainfall_lag_{lag}'] = price_ml_base.groupby(['market', 'commodity'])['rainfall_mm'].shift(lag)

price_ml_base['month'] = price_ml_base['date'].dt.month

features = [
    'market_encoded', 'crop_encoded', 'month', 'rainfall_mm', 'temp_mean_c', 'demand_index',
    'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_12',
    'rainfall_lag_1', 'rainfall_lag_2', 'rainfall_lag_3', 'rainfall_lag_12'
]

print(f"Core rows: {len(price_ml_base)} | Features: {len(features)}")


Core rows: 4333 | Features: 14


## 2. Per-Pair Tiered Horizons
Each pair's most recent data point is a different distance from today. A fixed 3-month horizon would land in the past for any pair whose data is already more than 3 months old.

In [ ]:
latest_rows = price_ml_base.loc[price_ml_base.groupby(['market', 'commodity'])['date'].idxmax()]
latest_rows = latest_rows.set_index(['market', 'commodity'])
gap_months = ((TODAY - latest_rows['date']).dt.days / 30.44).round().astype(int).clip(lower=1)
print(f"Distinct horizons needed: {sorted(gap_months.unique())}")


def evaluate_horizon(horizon):
    pm = price_ml_base.copy()
    pm['target_price'] = pm.groupby(['market', 'commodity'])['price'].shift(-horizon)
    lag_check = [f'price_lag_{l}' for l in [1,2,3,12]] + [f'rainfall_lag_{l}' for l in [1,2,3,12]]
    pm = pm.dropna(subset=lag_check + ['demand_index', 'target_price'])

    train_parts, test_parts = [], []
    for (mkt, crop), group in pm.groupby(['market', 'commodity']):
        group = group.sort_values('date')
        split_point = int(len(group) * 0.8)
        if split_point < 5 or len(group) - split_point < 2:
            continue
        train_parts.append(group.iloc[:split_point])
        test_parts.append(group.iloc[split_point:])
    train_df = pd.concat(train_parts)
    test_df = pd.concat(test_parts)

    overall_train_mean = train_df['price'].mean()
    market_means = train_df.groupby('market')['price'].mean()
    crop_means = train_df.groupby('commodity')['price'].mean()
    for d in [train_df, test_df]:
        d['market_encoded'] = d['market'].map(market_means).fillna(overall_train_mean)
        d['crop_encoded'] = d['commodity'].map(crop_means).fillna(overall_train_mean)

    model = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5,
                         random_state=42, verbosity=0)
    model.fit(train_df[features], train_df['target_price'])
    preds = pd.Series(model.predict(test_df[features]), index=test_df.index)

    win_pairs = set()
    for (mkt, crop), group in test_df.groupby(['market', 'commodity']):
        idx = group.index
        b_mape = mean_absolute_percentage_error(group['target_price'], group['price'])
        m_mape = mean_absolute_percentage_error(group['target_price'], preds.loc[idx])
        if m_mape < b_mape:
            win_pairs.add((mkt, crop))

    return {'model': model, 'win_pairs': win_pairs,
            'market_means': market_means, 'crop_means': crop_means,
            'overall_train_mean': overall_train_mean}


horizon_results = {int(h): evaluate_horizon(int(h)) for h in sorted(gap_months.unique())}
for h, r in horizon_results.items():
    print(f"Horizon {h}mo: {len(r['win_pairs'])} / 30 pairs favor the model")


Distinct horizons needed: [np.int64(3), np.int64(7), np.int64(8), np.int64(15)]
Horizon 3mo: 9 / 30 pairs favor the model
Horizon 7mo: 17 / 30 pairs favor the model
Horizon 8mo: 20 / 30 pairs favor the model
Horizon 15mo: 17 / 30 pairs favor the model


## 3. Live Forecast, Anchored to Today


In [ ]:
forecast_rows = []
for (mkt, crop), g in price_ml_base.groupby(['market', 'commodity']):
    g = g.sort_values('date')
    latest = g.iloc[-1]
    current_price = latest['price']
    as_of_date = latest['date']
    h = int(gap_months.loc[(mkt, crop)])
    hr = horizon_results[h]
    target_date = as_of_date + pd.DateOffset(months=h)

    if (mkt, crop) in hr['win_pairs']:
        feat = pd.DataFrame([{
            'market_encoded': hr['market_means'].get(mkt, hr['overall_train_mean']),
            'crop_encoded': hr['crop_means'].get(crop, hr['overall_train_mean']),
            'month': latest['month'],
            'rainfall_mm': latest['rainfall_mm'],      # real reference-date weather
            'temp_mean_c': latest['temp_mean_c'],      # real reference-date weather
            'demand_index': latest['demand_index'],
            'price_lag_1': current_price,
            'price_lag_2': g.iloc[-2]['price'] if len(g) >= 2 else current_price,
            'price_lag_3': g.iloc[-3]['price'] if len(g) >= 3 else current_price,
            'price_lag_12': g.iloc[-12]['price'] if len(g) >= 12 else current_price,
            'rainfall_lag_1': latest['rainfall_mm'],
            'rainfall_lag_2': g.iloc[-2]['rainfall_mm'] if len(g) >= 2 else latest['rainfall_mm'],
            'rainfall_lag_3': g.iloc[-3]['rainfall_mm'] if len(g) >= 3 else latest['rainfall_mm'],
            'rainfall_lag_12': g.iloc[-12]['rainfall_mm'] if len(g) >= 12 else latest['rainfall_mm'],
        }])
        forecast_price = hr['model'].predict(feat[features])[0]
        source = 'model'
    else:
        forecast_price = current_price
        source = 'baseline'

    forecast_rows.append({
        'market': mkt, 'commodity': crop, 'as_of_date': as_of_date,
        'target_date': target_date, 'data_age_months': h,
        'current_price': current_price, 'forecast_price': round(forecast_price, 1),
        'forecast_source': source,
    })

forecast_df = pd.DataFrame(forecast_rows)
print(forecast_df.to_string(index=False))
print(f"\nSource distribution: {forecast_df['forecast_source'].value_counts().to_dict()}")


 market     commodity as_of_date target_date  data_age_months  current_price  forecast_price forecast_source
 Iganga         Beans 2025-12-15  2026-07-15                7         3289.0     3375.300049           model
 Iganga Cassava Flour 2025-12-15  2026-07-15                7         1300.0     1300.000000        baseline
 Iganga Maize (White) 2025-12-15  2026-07-15                7         1533.0     1212.099976           model
 Iganga   Maize Flour 2025-12-15  2026-07-15                7         2078.0     2118.399902           model
 Iganga        Millet 2025-12-15  2026-07-15                7         2856.0     2856.000000        baseline
 Iganga       Sorghum 2025-12-15  2026-07-15                7         1844.0     1901.599976           model
  Jinja         Beans 2026-04-15  2026-07-15                3         4211.0     4211.000000        baseline
  Jinja Cassava Flour 2026-04-15  2026-07-15                3         1724.0     1724.000000        baseline
  Jinja Maize (Whit

## 4. Sell-Now-vs-Wait Signal

In [ ]:
CONFIDENCE_THRESHOLD = 0.03

forecast_df['pct_change'] = (
    (forecast_df['forecast_price'] - forecast_df['current_price']) / forecast_df['current_price']
)

def sell_signal(pct):
    if pct > CONFIDENCE_THRESHOLD: return 'wait'
    elif pct < -CONFIDENCE_THRESHOLD: return 'sell_now'
    else: return 'hold'

forecast_df['sell_signal'] = forecast_df['pct_change'].apply(sell_signal)
forecast_df['sell_confidence'] = forecast_df['pct_change'].abs().clip(upper=0.3) / 0.3

print(forecast_df[['market', 'commodity', 'data_age_months', 'sell_signal',
                    'sell_confidence', 'forecast_source']].to_string(index=False))
print(f"\nSignal distribution: {forecast_df['sell_signal'].value_counts().to_dict()}")


 market     commodity  data_age_months sell_signal  sell_confidence forecast_source
 Iganga         Beans                7        hold         0.087463           model
 Iganga Cassava Flour                7        hold         0.000000        baseline
 Iganga Maize (White)                7    sell_now         0.697760           model
 Iganga   Maize Flour                7        hold         0.064806           model
 Iganga        Millet                7        hold         0.000000        baseline
 Iganga       Sorghum                7        wait         0.104121           model
  Jinja         Beans                3        hold         0.000000        baseline
  Jinja Cassava Flour                3        hold         0.000000        baseline
  Jinja Maize (White)                7        hold         0.000000        baseline
  Jinja   Maize Flour                3        hold         0.000000        baseline
  Jinja        Millet                7    sell_now         0.149556         

## 5. Arbitrage / Best Market Finder
Uses the wide dataset (all markets, not just the 5-market core).

In [ ]:
core_crops = sorted(price_ml_base['commodity'].unique())
wide_core = data[data['commodity'].isin(core_crops)]

latest_per_pair = wide_core.loc[wide_core.groupby(['market', 'commodity'])['date'].idxmax()]
best_market = (
    latest_per_pair.loc[latest_per_pair.groupby('commodity')['price'].idxmax()]
    [['commodity', 'market', 'price']]
    .rename(columns={'market': 'best_market', 'price': 'best_price'})
)
arbitrage_df = latest_per_pair.merge(best_market, on='commodity')
arbitrage_df['price_gap_pct'] = round(
    100 * (arbitrage_df['best_price'] - arbitrage_df['price']) / arbitrage_df['price'], 1
)
print(arbitrage_df[['market', 'commodity', 'best_market', 'price_gap_pct']]
      .sort_values('price_gap_pct', ascending=False).head(10).to_string(index=False))


   market     commodity best_market  price_gap_pct
Makaratin Cassava Flour        Gulu          686.6
    Napak       Sorghum      Mukono          429.6
   Kiboga       Sorghum      Mukono          423.0
Makaratin       Sorghum      Mukono          415.9
    Napak Cassava Flour        Gulu          370.0
     Lira       Sorghum      Mukono          354.8
Nabilatuk Cassava Flour        Gulu          330.2
   Kasese Cassava Flour        Gulu          324.4
   Kotido       Sorghum      Mukono          306.2
  Karenga       Sorghum      Mukono          300.8


## 6. Demand Level
Read directly from `demand_index` in the dataset.
Demand is classified into Low / Moderate / High using per-crop terciles on the forecastable rows, matching how the column was designed.

In [ ]:
core_data = data[data['commodity'].isin(core_crops) & data['forecastable']]

def classify_demand(group):
    q1, q3 = group['demand_index'].quantile([0.33, 0.66])
    def label(val):
        if val <= q1: return 'Low'
        elif val >= q3: return 'High'
        else: return 'Moderate'
    return group['demand_index'].apply(label)

core_data = core_data.copy()
core_data['demand_level'] = core_data.groupby('commodity', group_keys=False).apply(classify_demand)

latest_demand = (
    core_data.loc[core_data.groupby(['market', 'commodity'])['date'].idxmax()]
    [['market', 'commodity', 'demand_index', 'demand_level']]
)
print(latest_demand.to_string(index=False))


 market     commodity  demand_index demand_level
 Iganga         Beans      0.499806     Moderate
 Iganga Cassava Flour      0.684032     Moderate
 Iganga Maize (White)      0.441706          Low
 Iganga   Maize Flour      0.581354     Moderate
 Iganga        Millet      0.462448          Low
 Iganga       Sorghum      0.462882     Moderate
  Jinja         Beans      0.389424          Low
  Jinja Cassava Flour      0.514110          Low
  Jinja Maize (White)      0.364411          Low
  Jinja   Maize Flour      0.475357          Low
  Jinja        Millet      0.440504          Low
  Jinja       Sorghum      0.427238          Low
   Lira         Beans      0.470025          Low
   Lira Cassava Flour      0.406096          Low
   Lira Maize (White)      0.732461         High
   Lira   Maize Flour      0.693175         High
   Lira        Millet      0.527547     Moderate
   Lira       Sorghum      0.921853         High
Mbarara         Beans      0.454409          Low
Mbarara Cassava Flou

## 7. Anomaly Flag
Flags the latest price as anomalous if its modified z-score > 3.5 against the pair's own history.

In [ ]:
price_ml_full = price_ml_base.copy()
anomaly_rows = []
for (mkt, crop), g in price_ml_full.groupby(['market', 'commodity']):
    g = g.sort_values('date')
    med = g['price'].median()
    mad = (g['price'] - med).abs().median()
    latest_price = g['price'].iloc[-1]
    if mad == 0 or pd.isna(mad):
        flag = False
    else:
        flag = abs(0.6745 * (latest_price - med) / mad) > 3.5
    anomaly_rows.append({'market': mkt, 'commodity': crop, 'anomaly_flag': flag})
anomaly_df = pd.DataFrame(anomaly_rows)
print(f"Anomalies flagged: {anomaly_df['anomaly_flag'].sum()} / {len(anomaly_df)}")


Anomalies flagged: 0 / 30


## 8. Combined Decision Output

In [ ]:
decision_outputs = forecast_df[[
    'market', 'commodity', 'as_of_date', 'target_date', 'data_age_months',
    'current_price', 'forecast_price', 'forecast_source', 'sell_signal', 'sell_confidence'
]].copy()

arb_slim = arbitrage_df[[
    'market', 'commodity', 'best_market', 'price_gap_pct'
]]
decision_outputs = decision_outputs.merge(arb_slim, on=['market', 'commodity'], how='left')
decision_outputs = decision_outputs.merge(
    latest_demand[['market', 'commodity', 'demand_index', 'demand_level']],
    on=['market', 'commodity'], how='left'
)
decision_outputs = decision_outputs.merge(anomaly_df, on=['market', 'commodity'], how='left')

print(decision_outputs.to_string(index=False))


 market     commodity as_of_date target_date  data_age_months  current_price  forecast_price forecast_source sell_signal  sell_confidence                    best_market  price_gap_pct  demand_index demand_level  anomaly_flag
 Iganga         Beans 2025-12-15  2026-07-15                7         3289.0     3375.300049           model        hold         0.087463                         Kasese           52.0      0.499806     Moderate         False
 Iganga Cassava Flour 2025-12-15  2026-07-15                7         1300.0     1300.000000        baseline        hold         0.000000                           Gulu          189.2      0.684032     Moderate         False
 Iganga Maize (White) 2025-12-15  2026-07-15                7         1533.0     1212.099976           model    sell_now         0.697760                        Mbarara           60.9      0.441706          Low         False
 Iganga   Maize Flour 2025-12-15  2026-07-15                7         2078.0     2118.399902        

## 9. Save

In [ ]:
from pathlib import Path
from google.colab import files

out_dir = Path.home() / 'Desktop'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'decision_outputs.csv'
decision_outputs.to_csv(out_path, index=False)
print(f"Saved: {out_path}  |  shape: {decision_outputs.shape}")
print(f"Columns: {decision_outputs.columns.tolist()}")

files.download(out_path)

Saved: /root/Desktop/decision_outputs.csv  |  shape: (30, 15)
Columns: ['market', 'commodity', 'as_of_date', 'target_date', 'data_age_months', 'current_price', 'forecast_price', 'forecast_source', 'sell_signal', 'sell_confidence', 'best_market', 'price_gap_pct', 'demand_index', 'demand_level', 'anomaly_flag']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>